# How to use the oneClick-ML utility in the Atlas cluster

## Import `digitalhub` library and create a (or get an existing) project

In [1]:
import digitalhub as dh

Thie following method works both if the project has already been created or not

In [2]:
project = dh.get_or_create_project('oneclickml-atlas')

## Load input dataset

Load the input data as artifact or dataitem. The two files in this example are named `data.csv` and `config.ini`. After that, it can be found on the "Data items" and "Artifact" section of the Core managment UI

In [3]:
csv_data = project.log_dataitem(
    name='csv_input_data' ,
    kind='table' ,
    source='oneClick-ML/data.csv'
)
csv_data.key

/home/anaclerio/python3.12/lib/python3.12/site-packages/digitalhub/entities/dataitem/crud.py:146: UserWarning: log_dataitem in version 0.16 will log a dataitem of kind 'dataitem'. To log a dataitem of a specific kind, use the log_<kind> methods (e.g. log_table).
  warn(


'store://oneclickml-atlas/dataitem/table/csv_input_data:75532d5c73f94a7783af41b7c30926a1'

In [4]:
config_data = project.log_artifact(
    name='config_ini_file' ,
    kind='artifact' ,
    source='oneClick-ML/config.ini'
)
config_data.key

/home/anaclerio/python3.12/lib/python3.12/site-packages/digitalhub/entities/artifact/crud.py:134: UserWarning: log_artifact in version 0.16 wil log an artifact of kind 'artifact'.
  warn("log_artifact in version 0.16 wil log an artifact of kind 'artifact'.")
/home/anaclerio/python3.12/lib/python3.12/site-packages/digitalhub/entities/artifact/artifact/crud.py:61: UserWarning: This method will become log_artifact in 0.16
  warn("This method will become log_artifact in 0.16")


'store://oneclickml-atlas/artifact/artifact/config_ini_file:b4c28dfe904640cb91919541b7f8d520'

## General overview on how the functions work

A function first needs to be declared, then needs to be builded, and after these steps can be executed ("runned") on different data multiple times.

If it's the first time that the function is used, first register it by running `Cell 1`, then build it by running `Cell 2` and then execute it by running `Cell 5`. It can be re-executed on different data by changing input and output data in the options `input` and `parameters`.

If the function has already been declared and builded, and:
* the notebook has been reopened (= the jupyter kernel has been restarted), rerun the first cells, after that `Cell 2` and `Cell 5`
* the code has been modified, run `Cell 2` to get the function, then `Cell 4` to fetch the build and `Cell 1` ***with the parameter `image` decommented***; finally run directly `Cell 5`

## Declaration of the main helper function

In [ ]:
# Cell 1 - declares a function

main_func = project.new_function(
    name="main_function",
    kind="python",
    python_version="PYTHON3_10",
    requirements="oneClick-ML/requirements.txt",
    code_src="oneClick-ML", # this is the folder where all the original repo files are cloned
    handler="main:helper_main" # main is the original main.py from the repo : train is the new function (that calls the original main function)
    # image=main_func_ref,
)

If it has already been declared, just get it by pasting the id (Core managment UI > Functions > Click on functions > id). It's possible to retrieve multiple versions of a function.

In [ ]:
# Cell 2 - retrieves an already declared function

# main_func = project.get_function("main_function",
#     entity_id=""
# )

## Building of the main helper function

In [ ]:
# Cell 3 - builds the function

main_func_build = main_func.run(action="build",
    resources={"cpu": "4", "mem": "16Gi", "disk": "10Gi"},
)

In [ ]:
# Cell 4 - gets the reference of the function

main_func_ref = main_func.refresh().spec.image
print(main_func_ref)

None


## Running of the main helper function

The execution can be monitored in the "Runs" section of the Core managment UI, while the result can be found in the "Artifacts" section.

In [ ]:
# Cell 5 - runs a function

main_func_run = main_func.run(action='job',
    inputs={
        'csv_file': csv_data.key ,
        'config_file': config_data.key
    },
    parameters={
        # Place here the name of the output artifact
        'output_artifact_name': 'output_data' ,
    },
    resources={
        "cpu": "16",
        "mem": "32Gi",
        "disk": "50Gi",
    },
    execution=False,
)